In [0]:
%pip install python-pptx Pillow openai -q

In [0]:
# ── Configuration ──
# Update these values if needed
VOLUME_PATH = "/Volumes/integrated_intelligence_poc/default/kiip_docs/ppts"
APP_DIR = "/Workspace/Users/malaiarasu.g@vcreatek.com/chatbot-api"

# Azure OpenAI settings (same as app.yaml)
AZURE_ENDPOINT = "https://databricks-api.openai.azure.com/"
DEPLOYMENT = "gpt-4o"
API_VERSION = "2024-10-21"
API_KEY = dbutils.secrets.get(scope="your-scope", key="azure-openai-api-key")  # <-- Configure your Databricks secret scope

In [0]:
import os, json, hashlib, base64, logging
from pptx import Presentation
from openai import AzureOpenAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=API_KEY, api_version=API_VERSION)

# ── Extraction helpers ──
def extract_text(slide):
    texts = []
    for shape in slide.shapes:
        if shape.has_table: continue
        if shape.has_text_frame:
            for p in shape.text_frame.paragraphs:
                t = p.text.strip()
                if t: texts.append(t)
    return "\n".join(texts)

def extract_tables(slide):
    tables = []
    for shape in slide.shapes:
        if shape.has_table:
            rows = [" | ".join(c.text.strip() for c in row.cells) for row in shape.table.rows]
            tables.append("\n".join(rows))
    return "\n\n".join(tables)

def extract_images(slide):
    imgs = []
    for shape in slide.shapes:
        if hasattr(shape, "image"):
            try:
                imgs.append((shape.image.blob, shape.image.content_type or "image/png"))
            except: pass
    return imgs

def describe_image(img_bytes, content_type):
    b64 = base64.b64encode(img_bytes).decode("utf-8")
    try:
        resp = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[
                {"role": "system", "content": "Describe this image from a presentation slide concisely. Focus on key information, data, labels, and any text visible."},
                {"role": "user", "content": [{"type": "image_url", "image_url": {"url": f"data:{content_type};base64,{b64}", "detail": "high"}}]}
            ],
            max_tokens=500,
        )
        return resp.choices[0].message.content
    except Exception as e:
        logger.warning(f"Image failed: {e}")
        return "[Image could not be described]"

def generate_summary(ppt_data):
    slides_text = ""
    for s in ppt_data["slides"]:
        slides_text += f"\n--- Slide {s['slide_number']} ---\n{s['content']}\n"
    resp = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a document analyst. Create a concise summary and table of contents for this PowerPoint. Include main topics, key points, and sections. This will be used to match user queries."},
            {"role": "user", "content": f"Presentation: {ppt_data['filename']}\n\n{slides_text}"}
        ],
        temperature=0.3, max_tokens=1000,
    )
    return resp.choices[0].message.content

# ── Process all PPTs ──
file_hashes = {}
ppt_data_list = []
summaries = []

for fname in sorted(os.listdir(VOLUME_PATH)):
    if not fname.lower().endswith(".pptx"): continue
    fpath = os.path.join(VOLUME_PATH, fname)
    h = hashlib.md5()
    with open(fpath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    file_hashes[fname] = h.hexdigest()

    print(f"\n{'='*60}")
    print(f"Processing: {fname}")
    prs = Presentation(fpath)
    slides = []

    for idx, slide in enumerate(prs.slides, 1):
        text = extract_text(slide)
        tables = extract_tables(slide)
        images = extract_images(slide)
        img_descs = [describe_image(b, ct) for b, ct in images]

        parts = []
        if text: parts.append(f"Text:\n{text}")
        if tables: parts.append(f"Tables:\n{tables}")
        if img_descs: parts.append("Images:\n" + "\n".join(img_descs))
        combined = "\n\n".join(parts) if parts else "[Empty slide]"
        slides.append({"slide_number": idx, "content": combined})
        print(f"  Slide {idx}: {len(images)} images | {len(text)} chars text")

    data = {"filename": fname, "file_path": fpath, "total_slides": len(slides), "slides": slides}
    ppt_data_list.append(data)

    summary = generate_summary(data)
    summaries.append({"filename": fname, "summary": summary})
    print(f"  \u2705 {len(slides)} slides indexed | Summary generated")

print(f"\n{'='*60}")
print(f"Total presentations processed: {len(ppt_data_list)}")

In [0]:
# Build cache
cache = {
    "file_hashes": file_hashes,
    "summaries": summaries,
    "ppt_data": ppt_data_list,
}

# Save to the app deployment directory (deploys with the app)
cache_path = os.path.join(APP_DIR, "ppt_cache.json")
with open(cache_path, "w") as f:
    json.dump(cache, f, indent=2)

size_kb = os.path.getsize(cache_path) / 1024
print(f"\u2705 Cache saved to: {cache_path}")
print(f"   Size: {size_kb:.1f} KB")
print(f"   Presentations: {len(ppt_data_list)}")
for s in summaries:
    print(f"   \u2022 {s['filename']}")
print(f"\n\U0001f680 Now redeploy the app \u2014 the cache will be included automatically.")